# Simple Chat Agent (Pure Python) - Microsoft Foundry Hosted Agent 배포

이 노트북은 순수 Python으로 구현한 `simple-chat` 에이전트를 Microsoft Foundry에 Hosted Agent로 배포합니다.

**langgraph 버전과의 차이:** LangGraph/LangChain 없이 OpenAI SDK + Starlette만으로 Responses API 인터페이스를 직접 구현합니다.

**사전 준비 (최초 1회):**
- **Capability Host 생성** — Account 수준에서 `capabilityHostKind: Agents` + `enablePublicHostingEnvironment: true`로 생성
- **RBAC 권한 부여** — Foundry 프로젝트의 Managed Identity에 `AcrPull` + `Azure AI User` 역할 부여
- **모델 배포** — Foundry 프로젝트에 사용할 모델(예: `gpt-4.1`) 배포 완료
- **`.env` 작성** — `.env.example`을 복사하여 `.env`를 만들고 값 채우기

**참고 문서:**
- [Deploy a hosted agent](https://learn.microsoft.com/azure/foundry/agents/how-to/deploy-hosted-agent)
- [Manage hosted agent lifecycle](https://learn.microsoft.com/azure/foundry/agents/how-to/manage-hosted-agent)

---

# 공통: 환경 설정 & 로컬 테스트

## 0. 환경 변수 로드

`.env.example`을 복사하여 `.env`를 만들고 값을 채워주세요.

```bash
cp .env.example .env
```

노트북에서 사용할 라이브러리를 설치합니다.

In [ ]:
%pip install --quiet python-dotenv "azure-ai-projects>=2.0.0" azure-identity

In [ ]:
import os
import json
import datetime
from dotenv import load_dotenv

load_dotenv()

SUBSCRIPTION_ID = os.getenv("SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
ACCOUNT_NAME = os.getenv("ACCOUNT_NAME")
ACR_NAME = os.getenv("ACR_NAME")

AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AZURE_AI_MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")

print(f"Subscription: {SUBSCRIPTION_ID[:8]}...")
print(f"Resource Group: {RESOURCE_GROUP}")
print(f"Account Name: {ACCOUNT_NAME}")
print(f"Project Endpoint: {AZURE_AI_PROJECT_ENDPOINT[:20]}...")
print(f"OpenAI Endpoint: {AZURE_OPENAI_ENDPOINT[:20]}...")
print(f"Model: {AZURE_AI_MODEL_DEPLOYMENT_NAME}")
print(f"ACR: {ACR_NAME}")

## 1. 로컬 테스트

Docker 빌드 전에 로컬에서 에이전트가 정상 동작하는지 확인합니다.

터미널에서 가상환경 활성화 후 환경변수 등록, 의존성 설치, 서버 실행:

```bash
source .venv/bin/activate
cd hosted-agents/pure-python/simple-chat
pip install -r requirements.txt

# 환경변수 등록 (.env 파일의 값을 직접 export)
export AZURE_OPENAI_ENDPOINT="https://<your-resource>.openai.azure.com/"
export AZURE_AI_MODEL_DEPLOYMENT_NAME="gpt-4.1"
export AZURE_AI_PROJECT_ENDPOINT="https://<your-account>.services.ai.azure.com/api/projects/<your-project>"
```

> 💡 또는 `.env` 파일을 이용하여 한 번에 등록할 수 있습니다:
> ```bash
> export $(grep -v '^#' .env | xargs)
> python main.py
> ```

```bash
python main.py
```

서버가 `localhost:8088`에서 실행되면 아래 셀로 테스트합니다.

In [ ]:
import requests
import json

response = requests.post(
    "http://localhost:8088/responses",
    headers={"Content-Type": "application/json"},
    json={"input": "시애틀이 어디있나요?", "stream": False}
)

print(json.dumps(response.json(), indent=2, ensure_ascii=False))

---
# Python SDK 기반 배포

`azure-ai-projects>=2.0.0` SDK를 사용하여 배포합니다.

### 2-1. Docker 이미지 빌드 & 푸시

아래 셀을 실행하여 이미지 이름을 확인한 후, 터미널에서 빌드 & 푸시합니다.

> ⚠️ Apple Silicon 맥북에서는 `--platform=linux/amd64` 필수

In [ ]:
AGENT_NAME = "simple-chat-pure"
IMAGE_NAME = "simple-chat-pure"
# IMAGE_TAG = datetime.datetime.now().strftime("%Y%m%d%H%M")
IMAGE_TAG = 'v1'
FULL_IMAGE = f"{ACR_NAME}.azurecr.io/{IMAGE_NAME}:{IMAGE_TAG}"

print("터미널에서 아래 명령을 실행하세요:\n")
print(f"az login")
print(f"az acr login --name {ACR_NAME}")
print(f"docker build --platform=linux/amd64 -t {FULL_IMAGE} .")
print(f"docker push {FULL_IMAGE}")

### 2-2. Container Registry 권한 설정

Azure Portal에서 ACR에 Foundry 프로젝트의 `AcrPull` 권한을 부여합니다.

1. Azure Portal → **Container Registry** → 해당 ACR → **Access Control (IAM)**
2. **Add role assignment** → Role: `AcrPull`
3. Members → **Managed identity** 선택 → **Select members** → Microsoft Foundry 프로젝트 선택
4. **Review + assign**

### 2-3. Capability Host 생성

Hosted Agent 배포에 필수입니다. 이미 생성되어 있다면 건너뛰세요.

In [ ]:
!az rest --method put \
    --url "https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}/capabilityHosts/accountcaphost?api-version=2025-10-01-preview" \
    --headers "content-type=application/json" \
    --body '{{"properties": {{"capabilityHostKind": "Agents", "enablePublicHostingEnvironment": true}}}}'

### 2-4. Hosted Agent 생성

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)
from azure.identity import DefaultAzureCredential

project = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    description="A simple chat agent using pure Python (OpenAI SDK + Starlette)",
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="v1")
        ],
        cpu="1",
        memory="2Gi",
        image=FULL_IMAGE,
        environment_variables={
            "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
            "AZURE_AI_PROJECT_ENDPOINT": AZURE_AI_PROJECT_ENDPOINT,
            "AZURE_AI_MODEL_DEPLOYMENT_NAME": AZURE_AI_MODEL_DEPLOYMENT_NAME,
            "OPENAI_API_VERSION": "2025-03-01-preview",
        },
    ),
)

print(f"✅ Agent created: {agent.name}, version: {agent.version}")

### 2-5. Agent 테스트 (Python SDK)

In [ ]:
agent_info = project.agents.get(agent_name=AGENT_NAME)
print(f"Agent: {agent_info.name}")
print(f"Versions: {agent_info.versions}")

In [ ]:
openai_client = project.get_openai_client()

# 스트리밍 호출
stream = openai_client.responses.create(
    stream=True,
    input=[{"role": "user", "content": "Python의 장점을 간단히 설명해줘."}],
    extra_body={"agent_reference": {"name": agent_info.name, "type": "agent_reference"}},
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

### 2-6. 정리

In [ ]:
# Agent 버전 삭제
project.agents.delete_version(agent_name=AGENT_NAME, agent_version=agent.version)
print(f"Agent version {agent.version} deleted.")